# 03 · Evaluación de Modelos

Evalúa todos los modelos entrenados sobre un **test set** separado.

**Análisis incluidos:**
- Métricas por clase (Dice/IoU/F1/Hausdorff por capa retinal)
- Tiempo de inferencia y memoria GPU por modelo
- Análisis inter-rater: manual1 vs manual2 vs predicción (Duke)
- Guardado de predicciones para visualización

In [1]:
# ============================================================
#  CABECERA DE ENTORNO
# ============================================================
import sys, os
from pathlib import Path

def _detect_env():
    if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'): return 'kaggle'
    try:
        import google.colab; return 'colab'
    except ImportError: pass
    try: get_ipython(); return 'notebook'
    except NameError: return 'script'

ENV = _detect_env()
IS_NOTEBOOK = ENV in ('colab', 'kaggle', 'notebook')
print(f'[ENV] {ENV}')

if ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT_DIR = Path('/content/drive/MyDrive/oct-retina-segmentation')
elif ENV == 'kaggle':
    ROOT_DIR = Path('/kaggle/working')
else:
    _here = Path(globals().get('__file__', './')).resolve()
    ROOT_DIR = _here.parents[1] if _here.suffix == '.py' else _here.parent

DATA_DIR     = ROOT_DIR / 'data'
VARIANTS_DIR = DATA_DIR / 'Data_Kaggle'
CKPT_DIR     = ROOT_DIR / 'results' / 'checkpoints'
LOGS_DIR     = ROOT_DIR / 'logs'
EVAL_DIR     = ROOT_DIR / 'evaluation'
EVAL_DIR.mkdir(parents=True, exist_ok=True)
print(f'[ENV] CKPT_DIR = {CKPT_DIR}')
print(f'[ENV] EVAL_DIR = {EVAL_DIR}')

[ENV] notebook
[ENV] CKPT_DIR = C:\Users\User\oct-retina-segmentation\OCT\VC2\TP_vision_por_computadora_2\results\checkpoints
[ENV] EVAL_DIR = C:\Users\User\oct-retina-segmentation\OCT\VC2\TP_vision_por_computadora_2\evaluation


In [2]:
# ============================================================
#  IMPORTS
# ============================================================
import json, csv, time, warnings
warnings.filterwarnings('ignore')
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from scipy.ndimage import distance_transform_edt
import cv2
import pandas as pd

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[OK] Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'     GPU: {torch.cuda.get_device_name(0)}')

[OK] Device: cpu


In [3]:
# ============================================================
#  CONFIGURACION
# ============================================================
IMG_SIZE   = (256, 256)
BATCH_SIZE = 8
TEST_SIZE  = 0.2    # 20% reservado para test
SEED       = 42

# Modelos y variantes a evaluar
# Ajustar segun los checkpoints disponibles
EVAL_CONFIG = [
    # (modelo, variante, n_classes)
    ('unetpp',        'v1_kaggle_8c',      9),
]

# Nombres de capas para cada configuracion de clases
CLASS_NAMES_9 = ['Fondo sup.','ILM-RNFL','RNFL-GCL','GCL-INL',
                 'INL-OPL','OPL-ONL','ONL-IS','IS-RPE','Fondo inf.']

print(f'[OK] {len(EVAL_CONFIG)} combinaciones modelo × variante a evaluar.')

[OK] 1 combinaciones modelo × variante a evaluar.


---
## Bloque 1 · Dataset de test

In [4]:
class TestDataset(torch.utils.data.Dataset):
    """Dataset de test: carga variante .npy y reserva TEST_SIZE para evaluacion."""
    def __init__(self, variant_name, img_size=(256,256), clip_pct=(1.0,99.0)):
        self.img_size = img_size
        self.clip_pct = clip_pct
        if variant_name == 'v1_kaggle_8c':
            imgs  = np.load(VARIANTS_DIR / 'resized_images.npy')
            lbls  = np.load(VARIANTS_DIR / 'resized_labeledimages.npy').astype(np.int64)
        # Reservar 20% como test (mismo seed que entrenamiento para no contaminar)
        idx   = np.arange(len(imgs))
        _, test_idx = train_test_split(idx, test_size=TEST_SIZE,
                                       random_state=SEED)
        self.images = imgs[test_idx]
        self.labels = lbls[test_idx]
        print(f'  [{variant_name}] test={len(self.images)} muestras')

    def __len__(self): return len(self.images)

    def _normalize(self, img):
        p_lo, p_hi = np.percentile(img, self.clip_pct)
        img = np.clip(img, p_lo, p_hi)
        return ((img-p_lo)/max(p_hi-p_lo,1e-8)).astype(np.float32)

    def __getitem__(self, idx):
        img  = self.images[idx].astype(np.float32)
        mask = self.labels[idx].astype(np.float32)
        H, W = self.img_size
        img  = cv2.resize(img,  (W,H), interpolation=cv2.INTER_AREA)
        mask = cv2.resize(mask, (W,H), interpolation=cv2.INTER_NEAREST)
        img  = self._normalize(img)
        return {
            'image': torch.from_numpy(img).unsqueeze(0).float(),
            'mask':  torch.from_numpy(mask).long(),
            'idx':   idx,
        }

print('[OK] TestDataset.')

[OK] TestDataset.


---
## Bloque 2 · Definicion de modelos (igual que en entrenamiento)

In [5]:
# ---- U-Net ----
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch,out_ch,3,padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch,out_ch,3,padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class Down(nn.Module):
    def __init__(self,i,o): super().__init__(); self.net=nn.Sequential(nn.MaxPool2d(2),DoubleConv(i,o))
    def forward(self,x): return self.net(x)

class Up(nn.Module):
    def __init__(self,i,o):
        super().__init__()
        self.up=nn.Upsample(scale_factor=2,mode='bilinear',align_corners=True)
        self.conv=DoubleConv(i,o)
    def forward(self,x1,x2):
        x1=self.up(x1)
        dy,dx=x2.size(2)-x1.size(2),x2.size(3)-x1.size(3)
        x1=F.pad(x1,[dx//2,dx-dx//2,dy//2,dy-dy//2])
        return self.conv(torch.cat([x2,x1],dim=1))

class UNet(nn.Module):
    def __init__(self,n_channels=1,n_classes=9):
        super().__init__()
        self.inc=DoubleConv(n_channels,64); self.down1=Down(64,128)
        self.down2=Down(128,256); self.down3=Down(256,512); self.down4=Down(512,512)
        self.up1=Up(1024,256); self.up2=Up(512,128)
        self.up3=Up(256,64);   self.up4=Up(128,64)
        self.outc=nn.Conv2d(64,n_classes,1)
    def forward(self,x):
        x1=self.inc(x); x2=self.down1(x1); x3=self.down2(x2)
        x4=self.down3(x3); x5=self.down4(x4)
        x=self.up1(x5,x4); x=self.up2(x,x3)
        x=self.up3(x,x2);  x=self.up4(x,x1)
        return self.outc(x)

# ---- U-Net++ ----
class UpConcat(nn.Module):
    def __init__(self,i,o):
        super().__init__()
        self.up=nn.Upsample(scale_factor=2,mode='bilinear',align_corners=True)
        self.conv=DoubleConv(i,o)
    def forward(self,x_up,*skips):
        x=self.up(x_up); return self.conv(torch.cat([x,*skips],dim=1))

class UNetPP(nn.Module):
    def __init__(self,n_channels=1,n_classes=9):
        super().__init__()
        f=[64,128,256,512,1024]
        self.enc0=DoubleConv(n_channels,f[0])
        self.enc1=nn.Sequential(nn.MaxPool2d(2),DoubleConv(f[0],f[1]))
        self.enc2=nn.Sequential(nn.MaxPool2d(2),DoubleConv(f[1],f[2]))
        self.enc3=nn.Sequential(nn.MaxPool2d(2),DoubleConv(f[2],f[3]))
        self.enc4=nn.Sequential(nn.MaxPool2d(2),DoubleConv(f[3],f[4]))
        self.x01=UpConcat(f[1]+f[0],f[0]);    self.x11=UpConcat(f[2]+f[1],f[1])
        self.x21=UpConcat(f[3]+f[2],f[2]);    self.x31=UpConcat(f[4]+f[3],f[3])
        self.x02=UpConcat(f[1]+f[0]*2,f[0]);  self.x12=UpConcat(f[2]+f[1]*2,f[1])
        self.x22=UpConcat(f[3]+f[2]*2,f[2])
        self.x03=UpConcat(f[1]+f[0]*3,f[0]);  self.x13=UpConcat(f[2]+f[1]*3,f[1])
        self.x04=UpConcat(f[1]+f[0]*4,f[0])
        self.outc=nn.Conv2d(f[0],n_classes,1)
    def forward(self,x):
        x00=self.enc0(x); x10=self.enc1(x00); x20=self.enc2(x10)
        x30=self.enc3(x20); x40=self.enc4(x30)
        x01=self.x01(x10,x00); x11=self.x11(x20,x10)
        x21=self.x21(x30,x20); x31=self.x31(x40,x30)
        x02=self.x02(x11,x00,x01); x12=self.x12(x21,x10,x11)
        x22=self.x22(x31,x20,x21)
        x03=self.x03(x12,x00,x01,x02); x13=self.x13(x22,x10,x11,x12)
        x04=self.x04(x13,x00,x01,x02,x03)
        return self.outc(x04)

def build_model(model_name, n_classes):
    name = model_name.lower()
    if name == 'unet':    return UNet(1, n_classes)
    if name == 'unetpp':  return UNetPP(1, n_classes)
    if name == 'resnet_unet':
        import segmentation_models_pytorch as smp
        return smp.Unet(encoder_name='resnet34', encoder_weights=None,
                        in_channels=1, classes=n_classes)
    if name == 'segformer':
        from transformers import SegformerForSemanticSegmentation
        class SF(nn.Module):
            def __init__(self):
                super().__init__()
                self.model = SegformerForSemanticSegmentation.from_pretrained(
                    'nvidia/segformer-b2-finetuned-ade-512-512',
                    num_labels=n_classes, ignore_mismatched_sizes=True)
                old = self.model.segformer.encoder.patch_embeddings[0].proj
                self.model.segformer.encoder.patch_embeddings[0].proj = nn.Conv2d(
                    1, old.out_channels, old.kernel_size, old.stride, old.padding)
            def forward(self, x):
                l = self.model(pixel_values=x).logits
                return F.interpolate(l, size=x.shape[2:], mode='bilinear', align_corners=False)
        return SF()
    raise ValueError(f'Modelo no soportado: {model_name}')

print('[OK] Modelos definidos.')

[OK] Modelos definidos.


---
## Bloque 3 · Métricas

In [6]:
def compute_metrics_per_class(preds, targets, n_classes):
    """Calcula Dice/IoU/F1/Hausdorff por clase."""
    results = []
    for c in range(n_classes):
        p = (preds==c); t = (targets==c)
        inter = (p&t).sum(); union = (p|t).sum()
        dice  = float(2*inter/(p.sum()+t.sum()+1e-8))
        iou   = float(inter/(union+1e-8))
        # Hausdorff
        hd_vals = []
        for i in range(len(preds)):
            pi, ti = (preds[i]==c), (targets[i]==c)
            if pi.any() and ti.any():
                hd_vals.append(max(
                    distance_transform_edt(~ti)[pi].max(),
                    distance_transform_edt(~pi)[ti].max()
                ))
        hd = float(np.mean(hd_vals)) if hd_vals else 0.
        results.append({'class': c, 'dice': dice, 'iou': iou, 'f1': dice, 'hd': hd})
    return results

def measure_inference_time(model, device, n_runs=50, img_size=(256,256)):
    """Mide tiempo de inferencia promedio en ms/imagen."""
    model.eval()
    dummy = torch.randn(1, 1, *img_size).to(device)
    # Warmup
    with torch.no_grad():
        for _ in range(5): model(dummy)
    if device.type == 'cuda': torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        for _ in range(n_runs): model(dummy)
    if device.type == 'cuda': torch.cuda.synchronize()
    return (time.time()-t0)/n_runs*1000  # ms

def measure_gpu_memory(model, device, img_size=(256,256)):
    """Mide memoria GPU en MB durante inferencia."""
    if device.type != 'cuda': return 0.
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    dummy = torch.randn(1, 1, *img_size).to(device)
    model.eval()
    with torch.no_grad(): model(dummy)
    return torch.cuda.max_memory_allocated() / 1e6  # MB

print('[OK] compute_metrics_per_class / measure_inference_time / measure_gpu_memory.')

[OK] compute_metrics_per_class / measure_inference_time / measure_gpu_memory.


---
## Bloque 4 · Evaluación principal

In [7]:
all_eval_results = []
all_preds_store  = {}   # para visualizacion posterior

for model_name, variant_name, n_classes in EVAL_CONFIG:
    ckpt_path = CKPT_DIR / 'best_overall.pth'
    print ({ckpt_path})
    if not ckpt_path.exists():
        print(f'  [SKIP] {model_name}/{variant_name} — checkpoint no encontrado')
        continue

    print(f'\n--- {model_name} / {variant_name} ---')

    # Cargar modelo
    model = build_model(model_name, n_classes).to(DEVICE)
    ckpt  = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()

    # Dataset de test
    ds     = TestDataset(variant_name, img_size=IMG_SIZE)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    # Inferencia
    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch in loader:
            imgs  = batch['image'].to(DEVICE)
            masks = batch['mask']
            preds = torch.argmax(model(imgs), dim=1).cpu().numpy()
            all_preds.append(preds)
            all_targets.append(masks.numpy())

    preds_arr   = np.concatenate(all_preds)
    targets_arr = np.concatenate(all_targets)

    # Metricas por clase
    metrics_per_class = compute_metrics_per_class(preds_arr, targets_arr, n_classes)

    # Tiempo de inferencia y memoria
    infer_ms = measure_inference_time(model, DEVICE)
    gpu_mb   = measure_gpu_memory(model, DEVICE)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6

    print(f'  Inferencia: {infer_ms:.1f} ms/img  |  GPU: {gpu_mb:.0f} MB  |  Params: {n_params:.1f}M')

    class_names = CLASS_NAMES_9 if n_classes == 9 else CLASS_NAMES_7
    for m in metrics_per_class:
        c = m['class']
        print(f'  {class_names[c]:20s}  Dice={m["dice"]:.4f}  IoU={m["iou"]:.4f}  HD={m["hd"]:.2f}')

    # Guardar resultados
    for m in metrics_per_class:
        all_eval_results.append({
            'model': model_name, 'variant': variant_name,
            'class_idx': m['class'],
            'class_name': class_names[m['class']],
            'dice': round(m['dice'], 6), 'iou': round(m['iou'], 6),
            'f1':   round(m['f1'],   6), 'hd':  round(m['hd'],  4),
            'infer_ms': round(infer_ms, 2),
            'gpu_mb':   round(gpu_mb, 1),
            'n_params_M': round(n_params, 2),
        })

    # Guardar predicciones para visualizacion (primeras 20)
    key = f'{model_name}__{variant_name}'
    all_preds_store[key] = {
        'preds':   preds_arr[:20],
        'targets': targets_arr[:20],
        'imgs':    np.stack([ds.images[i] for i in range(min(20, len(ds)))]),
    }

    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print('\n[OK] Evaluacion completada.')

{WindowsPath('C:/Users/User/oct-retina-segmentation/OCT/VC2/TP_vision_por_computadora_2/results/checkpoints/best_overall.pth')}

--- unetpp / v1_kaggle_8c ---
  [v1_kaggle_8c] test=44 muestras
  Inferencia: 1345.6 ms/img  |  GPU: 0 MB  |  Params: 36.6M
  Fondo sup.            Dice=0.9961  IoU=0.9921  HD=2.79
  ILM-RNFL              Dice=0.9476  IoU=0.9004  HD=3.35
  RNFL-GCL              Dice=0.9642  IoU=0.9308  HD=3.36
  GCL-INL               Dice=0.9225  IoU=0.8562  HD=4.88
  INL-OPL               Dice=0.9103  IoU=0.8353  HD=6.32
  OPL-ONL               Dice=0.9736  IoU=0.9486  HD=4.58
  ONL-IS                Dice=0.9459  IoU=0.8974  HD=2.44
  IS-RPE                Dice=0.9348  IoU=0.8775  HD=2.54
  Fondo inf.            Dice=0.0000  IoU=0.0000  HD=0.00

[OK] Evaluacion completada.


---
## Bloque 6 · Guardar resultados

In [8]:
# Guardar CSV de metricas
df_eval = pd.DataFrame(all_eval_results)
df_eval.to_csv(EVAL_DIR / 'metricas_por_clase.csv', index=False)
print(f'[OK] metricas_por_clase.csv guardado ({len(df_eval)} filas)')

# Guardar predicciones para visualizacion
import pickle
with open(EVAL_DIR / 'predictions_sample.pkl', 'wb') as f:
    pickle.dump(all_preds_store, f)
print(f'[OK] predictions_sample.pkl guardado ({len(all_preds_store)} modelos)')

# Resumen rapido
print('\n=== RESUMEN DE EVALUACION ===')
summary = df_eval.groupby(['model','variant']).agg(
    dice_mean=('dice','mean'), iou_mean=('iou','mean'),
    hd_mean=('hd','mean'), infer_ms=('infer_ms','first'),
    gpu_mb=('gpu_mb','first')
).round(4).reset_index()
print(summary.to_string(index=False))
print('\n  → Siguiente: 04_resultados.ipynb')

[OK] metricas_por_clase.csv guardado (9 filas)
[OK] predictions_sample.pkl guardado (1 modelos)

=== RESUMEN DE EVALUACION ===
 model      variant  dice_mean  iou_mean  hd_mean  infer_ms  gpu_mb
unetpp v1_kaggle_8c     0.8439    0.8043   3.3622   1345.59     0.0

  → Siguiente: 04_resultados.ipynb
